<a href="https://colab.research.google.com/github/srikarthikB/personality-ml-model/blob/main/personality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install joblib
import pandas as pd
import re
import joblib
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
df1 = pd.read_csv('mbti_master_dataset.csv')
df2 = pd.read_csv('mbti_500.csv')
# print(df1.shape)
# print(df2.shape)

# print(df1.columns)
# print(df2.columns)

# common = set(df1['posts']) & set(df2['posts'])

# print("Overlap:", len(common))


In [5]:
mbti_types = [
    'infj', 'infp', 'intj', 'intp',
    'isfj', 'isfp', 'istj', 'istp',
    'enfj', 'enfp', 'entj', 'entp',
    'esfj', 'esfp', 'estj', 'estp'
]
def clean_text(text):
    text = text.lower()
    for personality in mbti_types:
        text = text.replace(personality, '')
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [6]:
df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=['posts', 'type'])
print(df.shape)

X = df['posts']
y = df['type']
X = X.astype(str)
X = X.apply(clean_text)
print("Text cleaning complete")
print("Total samples:", len(X))

(115582, 2)
Text cleaning complete
Total samples: 115582


In [7]:
# word_vectorizer = TfidfVectorizer(
#     analyzer='word',
#     ngram_range=(1,3),
#     stop_words='english',
#     max_features=30000
# )
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1,2),
    stop_words='english',
    max_features=15000
)

In [8]:
# char_vectorizer = TfidfVectorizer(
#     analyzer='char_wb',
#     ngram_range=(3,5),
#     max_features=30000
# )
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3,3),
    max_features=5000
)

In [9]:
print("Starting word vectorization...")
X_word = word_vectorizer.fit_transform(X)
print("Word vectorization complete")
print("Word matrix shape:", X_word.shape)

print("Starting char vectorization...")
X_char = char_vectorizer.fit_transform(X)
print("Char vectorization complete")
print("Char matrix shape:", X_char.shape)

X = hstack([X_word, X_char])
print("Combined matrix shape:", X.shape)

Starting word vectorization...
Word vectorization complete
Word matrix shape: (115582, 15000)
Starting char vectorization...
Char vectorization complete
Char matrix shape: (115582, 5000)
Combined matrix shape: (115582, 20000)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)
print("Train/test split complete")

Train/test split complete


In [11]:
model = LinearSVC(class_weight='balanced', C=1)
model.fit(X_train, y_train)
print("Model training complete")

predictions = model.predict(X_test)
print("Predictions complete")

Model training complete
Predictions complete


In [12]:
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.6551023056624995
              precision    recall  f1-score   support

        ENFJ       0.49      0.57      0.53       345
        ENFP       0.51      0.58      0.54      1368
        ENTJ       0.61      0.70      0.65       637
        ENTP       0.68      0.71      0.69      2482
        ESFJ       0.88      0.79      0.83        67
        ESFP       0.63      0.66      0.64       104
        ESTJ       0.82      0.79      0.81       126
        ESTP       0.85      0.90      0.87       437
        INFJ       0.66      0.66      0.66      3287
        INFP       0.61      0.62      0.62      2793
        INTJ       0.70      0.65      0.67      4704
        INTP       0.72      0.64      0.68      5253
        ISFJ       0.51      0.55      0.53       183
        ISFP       0.53      0.64      0.58       249
        ISTJ       0.47      0.56      0.51       310
        ISTP       0.58      0.72      0.64       772

    accuracy                           0.66     23117
   macr

In [13]:
joblib.dump(model, 'mbti_model.pkl')
joblib.dump(word_vectorizer, 'word_vectorizer.pkl')
joblib.dump(char_vectorizer, 'char_vectorizer.pkl')
print("Model and vectorizers saved!")

Model and vectorizers saved!
